# 🎬 Netflix Movie Recommendation System using SVD

This notebook builds a personalised movie recommendation engine using the **Netflix Prize dataset** and **SVD (Singular Value Decomposition)** matrix factorisation.

### Workflow:
1. Data Loading & Parsing
2. Exploratory Data Analysis (EDA)
3. Movie ID Mapping
4. Dataset Cleaning & Filtering
5. Load Movie Titles
6. SVD Model Training & Cross-Validation
7. Generate Personalised Recommendations
8. Conclusion

## 1. Import Libraries & Load Dataset

The Netflix Prize dataset stores ratings in a flat text file where **Movie IDs appear as row markers** (e.g., `1:`) rather than a separate column. We load it with only two columns: `Cust_Id` and `Rating`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')

df = pd.read_csv('combined_data_1.txt', names=['Cust_Id', 'Rating'], usecols=[0, 1]).copy()
df['Rating'] = df['Rating'].astype(float)

print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

## 2. Exploratory Data Analysis (EDA)

Before building a model, we explore the dataset to understand:
- How many unique movies and customers exist
- How ratings are distributed across 1–5 stars

In [ ]:
# Count movies, customers, and total ratings
movie_count  = df.isnull().sum()['Rating']           # NaN rows = movie ID markers
cust_count   = df['Cust_Id'].nunique() - movie_count
rating_count = df['Cust_Id'].count() - movie_count

print(f'Total Movies   : {movie_count}')
print(f'Total Customers: {cust_count}')
print(f'Total Ratings  : {rating_count}')

In [ ]:
# Rating distribution
ratings_count = df.groupby('Rating')['Rating'].agg(['count'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
ratings_count.plot(kind='barh', ax=axes[0], color='steelblue', legend=False)
axes[0].set_title('Rating Distribution (Count per Star)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Ratings')
axes[0].set_ylabel('Star Rating')

# Pie chart
axes[1].pie(
    ratings_count['count'],
    labels=[f'{int(r)}★' for r in ratings_count.index],
    autopct='%1.1f%%',
    colors=['#d73027','#fc8d59','#fee090','#91bfdb','#4575b4'],
    startangle=140
)
axes[1].set_title('Rating Share (%)', fontsize=13, fontweight='bold')

plt.suptitle(
    f'Total Pool: {movie_count} Movies | {cust_count} Customers | {rating_count} Ratings',
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.savefig('rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Movie ID Mapping

In the raw file, Movie IDs are embedded as NaN rows (e.g., `1:` appears as a row with no rating).
We extract those row positions and use them to assign the correct `Movie_Id` to every rating row.

In [ ]:
# Identify positions of movie ID marker rows (NaN ratings)
df_nan = pd.DataFrame(pd.isnull(df.Rating))
df_nan = df_nan[df_nan['Rating'] == True].reset_index()

print(f'Total movie ID markers found: {len(df_nan)}')

# Build Movie ID array
movie_np = []
movie_id = 1

for prev, curr in zip(df_nan['index'][:-1], df_nan['index'][1:]):
    movie_np.append(np.full(curr - prev - 1, movie_id))
    movie_id += 1

last_len = len(df) - df_nan.iloc[-1, 0] - 1
movie_np.append(np.full(last_len, movie_id))
movie_np = np.concatenate(movie_np)

print(f'Movie IDs assigned to {len(movie_np)} rating rows.')

## 4. Dataset Cleaning & Noise Filtering

Not all movies and customers are useful for building a good recommendation model:
- **Unpopular movies** (very few ratings) introduce noise
- **Inactive customers** (very few ratings given) are unreliable

We remove both below the **70th percentile** of their review counts.

In [ ]:
# Attach Movie IDs and drop NaN rows
dataset = df[pd.notnull(df['Rating'])].copy()
dataset['Movie_Id'] = movie_np.astype(int)
dataset['Cust_Id']  = dataset['Cust_Id'].astype(int)

print('Clean dataset shape:', dataset.shape)
print(dataset.head())

In [ ]:
f = ['count', 'mean']

# Movie filter — keep movies above 70th percentile of review count
dataset_movie_summary = dataset.groupby('Movie_Id')['Rating'].agg(f)
dataset_movie_summary.index = dataset_movie_summary.index.map(int)
movie_benchmark = round(dataset_movie_summary['count'].quantile(0.7), 0)
drop_movie_list = dataset_movie_summary[dataset_movie_summary['count'] < movie_benchmark].index
print(f'Movie minimum review threshold (70th percentile): {movie_benchmark}')
print(f'Movies to drop: {len(drop_movie_list)}')

# Customer filter — keep customers above 70th percentile of review count
dataset_cust_summary = dataset.groupby('Cust_Id')['Rating'].agg(f)
dataset_cust_summary.index = dataset_cust_summary.index.map(int)
cust_benchmark = round(dataset_cust_summary['count'].quantile(0.7), 0)
drop_cust_list = dataset_cust_summary[dataset_cust_summary['count'] < cust_benchmark].index
print(f'\nCustomer minimum review threshold (70th percentile): {cust_benchmark}')
print(f'Customers to drop: {len(drop_cust_list)}')

In [ ]:
# Apply filters
dataset = dataset[~dataset['Movie_Id'].isin(drop_movie_list)]
dataset = dataset[~dataset['Cust_Id'].isin(drop_cust_list)]

print(f'Dataset shape after filtering: {dataset.shape}')

# Pivot table — rows: customers, columns: movies, values: ratings
df_p = pd.pivot_table(dataset, values='Rating', index='Cust_Id', columns='Movie_Id')
print(f'User-Movie matrix shape: {df_p.shape}')

## 5. Load Movie Titles

We load `movie_titles.csv` to map Movie IDs to human-readable names, so recommendations are meaningful.

In [ ]:
df_title = pd.read_csv(
    'movie_titles.csv',
    encoding='ISO-8859-1',
    header=None,
    names=['Movie_Id', 'Year', 'Name'],
    on_bad_lines='skip'
)
df_title = df_title.set_index('Movie_Id')

print(f'Total movies in title list: {len(df_title)}')
print(df_title.head(10))

## 6. Install Dependencies

In [ ]:
!pip install scikit-surprise
!pip install "numpy<2"

## 7. SVD Model — Training & Cross-Validation

**SVD (Singular Value Decomposition)** is a matrix factorisation technique that learns latent features for both users and movies from the rating patterns. Instead of needing the full user-movie matrix, SVD decomposes it into lower-dimensional representations and learns what types of content each user prefers.

We use **3-fold cross-validation** and evaluate with **RMSE** (how far off predictions are on average) and **MAE** (average absolute error).

In [ ]:
import math
from surprise import Reader, Dataset, SVD
from surprise.model_selection import cross_validate

# Load 100,000 ratings into Surprise format
reader = Reader()
data   = Dataset.load_from_df(dataset[['Cust_Id', 'Movie_Id', 'Rating']][:100000], reader)

# Train SVD with 3-fold cross-validation
svd = SVD()
cv_results = cross_validate(svd, data, measures=['RMSE', 'MAE'], cv=3, verbose=True)

print(f'\n=== Cross-Validation Results ===')
print(f'Mean RMSE : {cv_results["test_rmse"].mean():.4f}')
print(f'Mean MAE  : {cv_results["test_mae"].mean():.4f}')

## 8. Generate Personalised Recommendations

We now:
1. Look up a specific user's existing 5-star movies (their known preferences)
2. Train SVD on the full dataset
3. Predict scores for all movies the user hasn't rated
4. Return the top-ranked recommendations

In [ ]:
# Step 1 — Show user's favourite movies (5-star rated)
USER_ID = 30878

user_favorites = dataset[(dataset['Cust_Id'] == USER_ID) & (dataset['Rating'] == 5)]
user_favorites = user_favorites.set_index('Movie_Id').join(df_title)['Name']

print(f"User {USER_ID}'s top-rated movies (5 stars):")
print(user_favorites.head(10).to_string())

In [ ]:
# Step 2 — Train SVD on the full dataset
data_full = Dataset.load_from_df(dataset[['Cust_Id', 'Movie_Id', 'Rating']], reader)
trainset  = data_full.build_full_trainset()
svd.fit(trainset)
print('Model trained on full dataset.')

In [ ]:
# Step 3 — Predict scores for all eligible movies
recommendations = df_title.copy().reset_index()
recommendations  = recommendations[~recommendations['Movie_Id'].isin(drop_movie_list)]
recommendations['Estimate_Score'] = recommendations['Movie_Id'].apply(
    lambda x: svd.predict(USER_ID, x).est
)
recommendations = recommendations.drop('Movie_Id', axis=1)
recommendations = recommendations.sort_values('Estimate_Score', ascending=False)

print(f'Top 10 Recommended Movies for User {USER_ID}:\n')
print(recommendations.head(10).to_string(index=False))

In [ ]:
# Visualise top 10 recommendations
top10 = recommendations.head(10)

plt.figure(figsize=(12, 5))
plt.barh(top10['Name'][::-1], top10['Estimate_Score'][::-1], color='steelblue')
plt.xlabel('Predicted Rating Score')
plt.title(f'Top 10 Movie Recommendations for User {USER_ID}', fontsize=13, fontweight='bold')
plt.xlim(3.5, 5.0)
plt.tight_layout()
plt.savefig('top10_recommendations.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Conclusion

| Step | Detail |
|------|--------|
| Dataset | Netflix Prize dataset (`combined_data_1.txt` + `movie_titles.csv`) |
| Total Ratings Processed | 100,000 (from a much larger pool) |
| Noise Filtering | 70th percentile threshold on both movies and customers |
| Algorithm | SVD (matrix factorisation via Surprise library) |
| Evaluation | 3-fold cross-validation — RMSE & MAE |
| Output | Ranked personalised movie recommendations with predicted scores |

### Key Learnings
- Real-world datasets are messy — the Netflix Prize format required a custom parsing pipeline before any modelling could happen
- Filtering inactive users and unpopular movies significantly improves model signal quality
- SVD learns latent preferences without needing explicit user profiles — it finds patterns purely from rating history
- Cross-validation gives a more reliable estimate of model accuracy than a single train/test split

---
**Author**: Karthikeyan K | B.Tech AI & Data Science | [GitHub](https://github.com/kt-keyan)